# This is a sample Jupyter Notebook

Below is an example of a code cell. 
Put your cursor into the cell and press Shift+Enter to execute it and select the next one, or click 'Run Cell' button.

Press Double Shift to search everywhere for classes, files, tool windows, actions, and settings.

To learn more about Jupyter Notebooks in PyCharm, see [help](https://www.jetbrains.com/help/pycharm/ipython-notebook-support.html).
For an overview of PyCharm, go to Help -> Learn IDE features or refer to [our documentation](https://www.jetbrains.com/help/pycharm/getting-started.html).

In [1]:
# 1. 导入依赖库

import pandas as pd

import numpy as np

import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split

from sklearn.tree import DecisionTreeClassifier, plot_tree

from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

from sklearn.preprocessing import LabelEncoder, StandardScaler

from sklearn.cluster import KMeans

from mlxtend.frequent_patterns import apriori, association_rules

import warnings

warnings.filterwarnings('ignore')

# 设置中文显示（解决中文乱码）

plt.rcParams['font.sans-serif'] = ['SimHei']  

plt.rcParams['axes.unicode_minus'] = False



# 2. 数据加载与合并（指定openpyxl引擎读取Excel）

try:

    # 读取四个表格（明确指定engine='openpyxl'，拖欠历史记录指定Sheet1）

    credit_df = pd.read_excel('数据-客户信用记录.xlsx', engine='openpyxl')  # 客户信用记录

    apply_df = pd.read_excel('数据-申请客户信息.xlsx', engine='openpyxl')    # 申请客户信息

    # 关键修改1：指定读取Sheet1，避免读取空Sheet

    default_df = pd.read_excel('数据-拖欠历史记录.xlsx', engine='openpyxl', sheet_name='Sheet1')  # 拖欠历史记录

    consume_df = pd.read_excel('数据-消费历史记录.xlsx', engine='openpyxl') # 消费历史记录

    print("数据读取成功！")

except Exception as e:

    print(f"数据读取失败：{str(e)}")

    print("请检查：1. 文件路径是否正确 2. 文件名是否与代码一致 3. openpyxl已安装")

    exit()



# 统一客户标识字段（兼容不同表格的字段名差异）

print("\n各表格客户标识字段：")

print(f"客户信用记录：{[col for col in credit_df.columns if '客户' in col]}")

print(f"申请客户信息：{[col for col in apply_df.columns if '客户' in col]}")

print(f"拖欠历史记录：{[col for col in default_df.columns if '客户' in col]}")

print(f"消费历史记录：{[col for col in consume_df.columns if '客户' in col]}")


 
# 统一为“客户号”（根据实际字段名调整）

for df in [apply_df, default_df, consume_df]:

    customer_cols = [col for col in df.columns if '客户' in col]

    if len(customer_cols) > 0 and customer_cols[0] != '客户号':

        df.rename(columns={customer_cols[0]: '客户号'}, inplace=True)



# 关键修改2：处理拖欠历史记录字段映射

# 表格字段：客户号、卡号、额度、拖欠标识、拖欠总金额、逾期天数

# 映射为代码需要的字段：拖欠金额、拖欠次数、逾期天数

default_df.rename(columns={

    '拖欠总金额': '拖欠金额',  # 映射总金额到拖欠金额

    '逾期天数': '逾期天数'     # 保留逾期天数字段（可作为额外特征）

}, inplace=True)



# 关键修改3：计算拖欠次数（按客户号计数，同一客户多条记录则累加）

default_df['拖欠次数'] = default_df.groupby('客户号')['拖欠标识'].transform('count')

# 若需去重（同一客户多次记录按1次计算），则使用：

# default_df['拖欠次数'] = default_df.groupby('客户号')['拖欠标识'].transform('nunique')



# 保留核心字段（剔除卡号等无关字段）

default_df = default_df[['客户号', '额度', '拖欠标识', '拖欠金额', '逾期天数', '拖欠次数']]



# 合并数据（左连接保留所有客户记录，避免数据丢失）

merge_df = pd.merge(credit_df, apply_df, on='客户号', how='left', suffixes=('_信用', '_申请'))

merge_df = pd.merge(merge_df, consume_df, on='客户号', how='left', suffixes=('', '_消费'))

merge_df = pd.merge(merge_df, default_df, on='客户号', how='left', suffixes=('', '_拖欠'))



# 3. 数据预处理（确保模型可用）

# 3.1 筛选核心特征（补充逾期天数字段）

core_features = [

    '客户号', '年龄_连续', '工作年限', '个人收入_连续', '信用总评分', '额度',

    '性别', '婚姻状态', '教育程度', '职业类型', '居住方式',

    '日均消费金额', '日均次数', '单笔消费最大金额',

    '拖欠次数', '拖欠金额', '逾期天数', '拖欠标识', '是否存在欺诈'

]

# 只保留存在的字段（避免部分表格缺少字段导致报错）

existing_features = [col for col in core_features if col in merge_df.columns]

df = merge_df[existing_features].copy()



# 3.2 处理缺失值

fill_cols = ['拖欠次数', '拖欠金额', '逾期天数', '拖欠标识', '是否存在欺诈']

for col in fill_cols:

    if col in df.columns:

        # 拖欠相关字段填充0（代表无拖欠记录）

        df[col] = df[col].fillna(0).astype(int)



# 3.3 分类型变量编码（标签编码，将文字转为数字）

label_cols = ['性别', '婚姻状态', '教育程度', '职业类型', '居住方式']

label_cols = [col for col in label_cols if col in df.columns]  # 只处理存在的字段

for col in label_cols:

    le = LabelEncoder()

    df[col] = le.fit_transform(df[col].astype(str))  # 转为字符串避免编码报错



# 3.4 定义目标变量（优化逻辑：基于拖欠标识或拖欠次数）

# 衍生拖欠标签：有拖欠标识（1）或拖欠次数>0均视为拖欠

df['是否拖欠'] = ((df['拖欠标识'] == 1) | (df['拖欠次数'] > 0)).astype(int) if '拖欠标识' in df.columns else 0

df['是否存在欺诈'] = df['是否存在欺诈'].fillna(0).astype(int)  # 确保欺诈标签无缺失# 4. 模型1：K-Means聚类（客户分群，识别高风险客户组）

# 4.1 选择聚类特征并标准化（增加逾期天数作为聚类特征）

cluster_features = ['年龄_连续', '个人收入_连续', '信用总评分', '日均消费金额', '逾期天数']

cluster_features = [col for col in cluster_features if col in df.columns]

if len(cluster_features) < 2:

    print("聚类特征不足，跳过聚类模型")

else:

    scaler = StandardScaler()

    df_scaled = scaler.fit_transform(df[cluster_features])

    # 4.2 肘部法则选择最优K值

    inertia = []

    k_range = range(2, 6)  # 缩小K值范围，加快运行速度

    for k in k_range:

        kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)  # n_init=10避免警告

        kmeans.fit(df_scaled)

        inertia.append(kmeans.inertia_)

    # 可视化肘部法则（保存图片用于实验报告）

    plt.figure(figsize=(8, 4))

    plt.plot(k_range, inertia, 'o-', color='orange', linewidth=2)

    plt.xlabel('聚类数量K')

    plt.ylabel('簇内平方和（惯性）')

    plt.title('K-Means最优K值选择（肘部法则）')

    plt.grid(alpha=0.3)

    plt.savefig('KMeans_肘部法则.png', dpi=300, bbox_inches='tight')

    plt.show()

    # 4.3 训练K-Means模型（K=4，适合业务分群）

    kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)

    df['客户聚类'] = kmeans.fit_predict(df_scaled)

    # 4.4 聚类结果分析（增加逾期天数指标）

    cluster_metrics = ['是否存在欺诈', '是否拖欠', '个人收入_连续', '信用总评分', '逾期天数', '拖欠金额']

    cluster_metrics = [col for col in cluster_metrics if col in df.columns]

    cluster_analysis = df.groupby('客户聚类')[cluster_metrics].mean().round(3)

    print("\n=== 聚类结果分析（客户分群风险特征） ===")

    print(cluster_analysis)



# 5. 模型2：决策树（欺诈/拖欠风险预测）

# 5.1 特征与目标变量拆分（剔除无关字段）

drop_cols = ['客户号', '是否存在欺诈', '是否拖欠', '拖欠次数', '拖欠金额', '拖欠标识', '客户聚类']

drop_cols = [col for col in drop_cols if col in df.columns]

X = df.drop(drop_cols, axis=1)

y_fraud = df['是否存在欺诈'] if '是否存在欺诈' in df.columns else np.zeros(len(df))

y_default = df['是否拖欠'] if '是否拖欠' in df.columns else np.zeros(len(df))



# 5.2 划分训练集/测试集（7:3拆分，保证样本分布均衡）

X_train_f, X_test_f, y_train_f, y_test_f = train_test_split(

    X, y_fraud, test_size=0.3, random_state=42, stratify=y_fraud if sum(y_fraud) > 0 else None

)

X_train_d, X_test_d, y_train_d, y_test_d = train_test_split(

    X, y_default, test_size=0.3, random_state=42, stratify=y_default if sum(y_default) > 0 else None

)



# 5.3 训练欺诈预测决策树（限制深度避免过拟合）

if sum(y_fraud) > 0:  # 有欺诈样本才训练

    dt_fraud = DecisionTreeClassifier(max_depth=5, random_state=42)

    dt_fraud.fit(X_train_f, y_train_f)

    y_pred_f = dt_fraud.predict(X_test_f)

    # 欺诈模型评估

    print("\n=== 欺诈预测决策树评估 ===")

    print(f"准确率：{accuracy_score(y_test_f, y_pred_f):.3f}")

    print("混淆矩阵：")

    print(confusion_matrix(y_test_f, y_pred_f))

    print("分类报告：")

    print(classification_report(y_test_f, y_pred_f, zero_division=0))

    # 可视化决策树（保存用于实验报告）

    plt.figure(figsize=(12, 8))

    plot_tree(dt_fraud, feature_names=X.columns, class_names=['无欺诈', '有欺诈'],

              filled=True, fontsize=7, rounded=True)

    plt.title('欺诈预测决策树模型')

    plt.savefig('欺诈预测决策树.png', dpi=300, bbox_inches='tight')

    plt.show()

else:

    print("\n无欺诈样本，跳过欺诈预测模型")



# 5.4 训练拖欠预测决策树

if sum(y_default) > 0:  # 有拖欠样本才训练

    dt_default = DecisionTreeClassifier(max_depth=5, random_state=42)

    dt_default.fit(X_train_d, y_train_d)

    y_pred_d = dt_default.predict(X_test_d)

    # 拖欠模型评估

    print("\n=== 拖欠预测决策树评估 ===")

    print(f"准确率：{accuracy_score(y_test_d, y_pred_d):.3f}")

    print("混淆矩阵：")

    print(confusion_matrix(y_test_d, y_pred_d))

    print("分类报告：")

    print(classification_report(y_test_d, y_pred_d, zero_division=0))

    # 可视化决策树

    plt.figure(figsize=(12, 8))

    plot_tree(dt_default, feature_names=X.columns, class_names=['无拖欠', '有拖欠'],

              filled=True, fontsize=7, rounded=True)

    plt.title('拖欠预测决策树模型')

    plt.savefig('拖欠预测决策树.png', dpi=300, bbox_inches='tight')

    plt.show()

else:

    print("\n无拖欠样本，跳过拖欠预测模型")  

数据读取失败：[Errno 2] No such file or directory: '数据-客户信用记录.xlsx'
请检查：1. 文件路径是否正确 2. 文件名是否与代码一致 3. openpyxl已安装

各表格客户标识字段：


NameError: name 'credit_df' is not defined

数据导入成功。

--- 缺失值填充前概览（未通过客户） ---
   信用总评分 信用等级  额度 审批结果_信用表
0    NaN  NaN NaN      NaN
1    NaN  NaN NaN      NaN


KeyError: "['客户姓名_x', '客户姓名_y'] not found in axis"

In [5]:
# -*- coding: utf-8 -*-
"""
银行信用卡欺诈与拖欠行为分析 — 完整脚本
说明:
- 请修改 'base_path' 变量以指向您的数据文件所在目录。
- 确保已安装所有必需的第三方包: 
  pip install pandas numpy matplotlib seaborn scikit-learn imbalanced-learn mlxtend openpyxl
"""

# ==================== 0. 导入库和环境设置 ====================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.svm import LinearSVC, SVC
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.cluster import KMeans
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
from imblearn.under_sampling import RandomUnderSampler
from imblearn.over_sampling import SMOTE
from mlxtend.frequent_patterns import apriori, association_rules
from IPython.display import display
import warnings
warnings.filterwarnings('ignore') # 忽略一些不重要的警告

sns.set(style="whitegrid") 

# --- 解决中文显示问题 ---
plt.rcParams['font.sans-serif'] = ['Microsoft YaHei', 'SimHei', 'KaiTi', 'Arial Unicode MS', 'sans-serif'] 
plt.rcParams['axes.unicode_minus'] = False 

# ------- 辅助函数：根据关键词查找列名（增强兼容性） -------
def find_column_by_keywords(df, keywords):
    """根据关键词列表查找 DataFrame 中的列名"""
    if df is None: return None
    for col in df.columns:
        for kw in keywords:
            if kw in col:
                return col
    return None

# -*- coding: utf-8 -*-
# (省略导入库和辅助函数 Find_column_by_keywords)

# ==================== 1. 步骤一：数据导入 (保持不变) ====================
print("="*40)
print("1. 步骤一：数据导入")
print("="*40)

# **【重要】请根据您的数据路径修改此变量**
base_path = r'E:/Demo/python/银行信用卡欺诈与拖欠行为分析/data/' 
data_tables = {}

try:
    df_credit = pd.read_excel(base_path + '数据-客户信用记录.xlsx')
    df_apply = pd.read_excel(base_path + '数据-申请客户信息.xlsx')
    df_consume = pd.read_excel(base_path + '数据-消费历史记录.xlsx')
    df_delinquency = pd.read_excel(base_path + '数据-拖欠历史记录.xlsx')
    data_tables = {'credit': df_credit, 'apply': df_apply, 'consume': df_consume, 'delinquency': df_delinquency}
    print("✅ 数据导入成功。")
except FileNotFoundError as e:
    print(f"🚨 错误：文件未找到。请检查路径是否正确：{base_path}")
    raise e

# ==================== 2. 步骤二：数据合并与清洗 (已修复) ====================
print("\n"+"="*40)
print("2. 步骤二：数据合并与清洗 (已修复 Index 问题)")
print("="*40)

# 1. 数据合并 
key_col = find_column_by_keywords(df_credit, ['客户号','customer','ID'])
if key_col is None: key_col = df_credit.columns[0]

df_merged = pd.merge(df_credit, df_apply, on=key_col, how='left', suffixes=('_credit', '_apply'))

# 2. 统一/重命名列（省略，假设已进行，这里聚焦 Index 修复）
col_map = {
    find_column_by_keywords(df_merged, ['年龄_credit', '年龄']): '年龄',
    find_column_by_keywords(df_merged, ['个人收入_连续_credit', '个人收入_连续']): '个人收入',
    # ... 其他列名映射（省略以节省空间）
    find_column_by_keywords(df_merged, ['审批结果']): '审批结果',
    find_column_by_keywords(df_merged, ['信用等级']): '信用等级',
}
df_merged.rename(columns={k:v for k,v in col_map.items() if k is not None}, inplace=True)

# 3. 清洗/填充
target_col_apply = '审批结果'
if target_col_apply in df_merged.columns:
    df_merged.dropna(subset=[target_col_apply], inplace=True)

# 填充缺失值（数值中位数，类别众数）
numerical_cols_to_fill = ['年龄', '个人收入', '额度', '信用总评分', '工作年限']
categorical_cols_to_fill = [
    '性别', '婚姻状态', '教育程度', '车辆情况',
    '户籍', '居住类型', '职业类别'
]

for col in numerical_cols_to_fill:
    if col in df_merged.columns:
        df_merged[col].fillna(df_merged[col].median(), inplace=True)
for col in categorical_cols_to_fill:
    if col in df_merged.columns:
        df_merged[col].fillna(df_merged[col].mode()[0], inplace=True)


# =========================================================================
# 【修复核心】在进行新列赋值前，重置 DataFrame 的 Index，确保索引唯一性。
# =========================================================================
if not df_merged.index.is_unique:
    df_merged.reset_index(drop=True, inplace=True)
    print("✅ 修复：检测到重复索引，已重置 DataFrame 索引。")
    
data_tables['merged'] = df_merged
print("✅ 数据合并与清洗完成。")


# ==================== 3. 步骤三：特征工程 (原出错位置) ====================
print("\n"+"="*40)
print("3. 步骤三：特征工程")
print("="*40)

df = data_tables['merged']

# 3.1 地区映射 (基于 '户籍' 列)
hukou_col = find_column_by_keywords(df, ['户籍'])
if hukou_col:
    region_map = {
        # ... 地区映射字典（省略）
        '黑龙江': '东北','辽宁': '东北','吉林': '东北', '山东':'华东','江苏':'华东','安徽':'华东','浙江':'华东','福建':'华东','上海':'华东',
        '广东':'华南','广西':'华南','海南':'华南', '湖北':'华中','湖南':'华中','河南':'华中','江西':'华中',
        '北京':'华北','天津':'华北','河北':'华北','山西':'华北','内蒙古':'华北', '四川':'西南','云南':'西南','贵州':'西南','西藏':'西南','重庆':'西南',
        '陕西':'西北','青海':'西北','甘肃':'西北','宁夏':'西北',
    }
    df['地区'] = df[hukou_col].astype(str).apply(lambda x: region_map.get(x, '其他'))
    print("✅ 衍生特征：'地区' 完成。")


# 3.2 年龄段与年收入等级 (基于 '年龄' 和 '个人收入' 列)
age_col = find_column_by_keywords(df, ['年龄'])
income_col = find_column_by_keywords(df, ['个人收入'])

def age_group(x):
    try: x = float(x)
    except: return '未知'
    if x <= 30: return '30岁以下'
    elif x <= 50: return '30-50岁'
    else: return '50岁以上'

def income_level(x):
    try: x = float(x)
    except: return 1
    if x <= 100000: return 1
    elif x <= 1000000: return 2
    elif x <= 10000000: return 3
    else: return 4

if age_col:
    # 这一行现在可以成功运行，因为 df_merged 的索引是唯一的
    df['年龄段'] = df[age_col].apply(age_group) 
    print("✅ 衍生特征：'年龄段' 完成。")
if income_col:
    df['年收入等级'] = df[income_col].apply(income_level)
    print("✅ 衍生特征：'年收入等级' 完成。")

data_tables['merged'] = df
print(df_consume.columns)
print(df_consume.index.is_unique)
# (后续步骤 4, 5, 6, 7 保持不变)

# ==================== 4. 步骤四：信用等级预测 (决策树) ====================
print("\n"+"="*40)
print("4. 步骤四：客户信用等级影响因素分析 (决策树)")
print("="*40)
grade_col = find_column_by_keywords(data_tables.get('merged'), ['信用等级'])

if 'merged' in data_tables and grade_col:
    df_grade = data_tables['merged'].copy()
    
    feature_candidates = [
        '年龄','性别','职业类别','年收入等级','车辆情况','保险缴纳','工作年限','居住类型'
    ]
    available_features = [c for c in feature_candidates if c in df_grade.columns]
    
    df_model = df_grade[available_features + [grade_col]].dropna(subset=[grade_col])
    
    # 准备特征 (编码)
    X = pd.get_dummies(df_model.drop(columns=[grade_col]), drop_first=True)
    y = df_model[grade_col]
    
    # 标准化数值特征
    scaler = StandardScaler()
    numeric_cols = X.select_dtypes(include=np.number).columns
    X[numeric_cols] = scaler.fit_transform(X[numeric_cols])

    if len(y.unique()) > 1 and len(X) > 10:
        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)
        
        # 决策树模型 (max_depth=5 为示例)
        tree = DecisionTreeClassifier(max_depth=5, random_state=42)
        tree.fit(X_train, y_train)
        y_pred = tree.predict(X_test)
        
        print("✅ 决策树模型 ('信用等级'预测) 结果:")
        print(f"准确率: {accuracy_score(y_test, y_pred):.4f}")
        print("\n分类报告:")
        print(classification_report(y_test, y_pred))
        
        # 特征重要性
        feature_importances = pd.Series(tree.feature_importances_, index=X.columns).sort_values(ascending=False).head(5)
        print("\n决策树特征重要性 Top 5 (结果应显示个人收入最重要):")
        print(feature_importances)
    else:
        print("⚠️ 无法运行 '信用等级' 预测：数据量不足或目标变量类别数少于 2。")


# ==================== 5. 步骤五：信用卡申请成功影响因素分析 (SVM/逻辑回归) ====================
print("\n"+"="*40)
print("5. 步骤五：信用卡申请成功影响因素分析 (SVM/逻辑回归)")
print("="*40)
target_col_apply = find_column_by_keywords(data_tables.get('merged'), ['审批结果'])

if 'merged' in data_tables and target_col_apply:
    df_apply_model = data_tables['merged'].copy()
    
    # 确保 '审批结果' 是 0/1 形式
    if df_apply_model[target_col_apply].dtype == object:
        df_apply_model[target_col_apply] = df_apply_model[target_col_apply].map(lambda x: 1 if '通过' in str(x) else 0)

    feature_candidates = [
        '年龄','性别','婚姻状态','职业类别','个人收入','工作年限','保险缴纳','车辆情况','教育程度'
    ]
    available_features = [c for c in feature_candidates if c in df_apply_model.columns]

    df_model = df_apply_model[available_features + [target_col_apply]]

    # 准备特征 (编码和缩放)
    X = pd.get_dummies(df_model.drop(columns=[target_col_apply]), drop_first=True)
    y = df_model[target_col_apply]
    
    # 标准化数值特征
    scaler = StandardScaler()
    numeric_cols = X.select_dtypes(include=np.number).columns
    X[numeric_cols] = scaler.fit_transform(X[numeric_cols])

    if y.nunique() == 2 and len(X) > 10:
        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)
        
        # --- 逻辑回归 (Logistic Regression) ---
        lr = LogisticRegression(max_iter=2000, random_state=42)
        lr.fit(X_train, y_train)
        y_pred_lr = lr.predict(X_test)
        print("--- 逻辑回归 结果 ---")
        print(f"准确率: {accuracy_score(y_test, y_pred_lr):.4f}")
        print(classification_report(y_test, y_pred_lr))
        
        # 特征重要性（系数）
        if hasattr(lr, 'coef_'):
            coefs = pd.Series(lr.coef_[0], index=X.columns).sort_values(key=abs, ascending=False).head(5)
            print("\nLogReg 特征重要性 Top 5 (绝对值排序):")
            print(coefs)
        
        # --- 线性 SVM (Linear SVC) ---
        svm = LinearSVC(C=1.0, max_iter=2000, random_state=42)
        svm.fit(X_train, y_train)
        y_pred_svm = svm.predict(X_test)
        print("\n--- 线性 SVM 结果 ---")
        print(f"准确率: {accuracy_score(y_test, y_pred_svm):.4f}")
        print(classification_report(y_test, y_pred_svm))

    else:
        print("⚠️ 无法运行 '审批结果' 预测：目标变量类别数不足 2 或数据量不足。")


# ==================== 6. 步骤六：信用卡欺诈检测与关联规则 ====================
print("\n"+"="*40)
print("6. 步骤六：信用卡欺诈检测与关联规则 (Apriori/SVM)")
print("="*40)

if 'consume' in data_tables and 'credit' in data_tables:
    df_consume = data_tables['consume'].copy()
    df_credit = data_tables['credit'].copy()
    
    # 6.1 数据合并与特征工程
    key_cons = find_column_by_keywords(df_consume, ['客户号','customer','ID'])
    key_cred = find_column_by_keywords(df_credit, ['客户号','customer','ID'])
    df_fraud = pd.merge(df_consume, df_credit[[key_cred, '额度', '个人收入']], left_on=key_cons, right_on=key_cred, how='left')
    
    # 统一列名（如果原始表中有）
    df_fraud.rename(columns={
        find_column_by_keywords(df_fraud, ['是否存在欺诈', '欺诈', 'fraud']): '是否存在欺诈',
        find_column_by_keywords(df_fraud, ['个人收入']): '个人收入',
        find_column_by_keywords(df_fraud, ['日均消费金额']): '日均消费金额',
        find_column_by_keywords(df_fraud, ['日均次数']): '日均次数',
    }, inplace=True)
    
    fraud_col = '是否存在欺诈'
    
    if fraud_col in df_fraud.columns:
        # 衍生特征：单笔是否透支 (假设最大消费高于额度)
        if '单笔消费最高金额' in df_fraud.columns and '额度' in df_fraud.columns:
            df_fraud['单笔是否透支'] = np.where(df_fraud['单笔消费最高金额'] > df_fraud['额度'], '超过', '未超过')
        # 衍生特征：刷卡频率
        if '日均次数' in df_fraud.columns:
            def freq(x):
                try: x = float(x)
                except: return '未知'
                if x <= 5: return '不频繁'
                elif x <= 10: return '频繁'
                else: return '非常频繁'
            df_fraud['刷卡频率'] = df_fraud['日均次数'].apply(freq)
        
        # 6.2 欺诈检测模型 (SVM)
        df_model = df_fraud.dropna(subset=[fraud_col]).copy()
        
        # 目标变量转换为 0/1 (假设 1/有/是 表示欺诈)
        if df_model[fraud_col].dtype == object:
             df_model['Fraud'] = df_model[fraud_col].map(lambda x: 1 if '1' in str(x) or '有' in str(x) or '是' in str(x) else 0)
        else:
             df_model['Fraud'] = df_model[fraud_col]
        
        X_cols = [c for c in ['额度','日均消费金额','日均次数','单笔消费最高金额','个人收入'] if c in df_model.columns]
        X = df_model[X_cols]
        y = df_model['Fraud']
        
        # 缺失值填充 (针对模型输入特征)
        X = X.fillna(X.median())
        
        # 标准化
        scaler = StandardScaler()
        X_scaled = scaler.fit_transform(X)
        
        # 类别不平衡处理：下采样非欺诈样本 (降低复杂度，根据报告采用此法)
        rus = RandomUnderSampler(sampling_strategy=0.2, random_state=42) 
        X_res, y_res = rus.fit_resample(X_scaled, y)
        
        X_train, X_test, y_train, y_test = train_test_split(X_res, y_res, test_size=0.3, random_state=42, stratify=y_res)
        
        # SVM 分类器 (核函数 'rbf' 为示例)
        svm = SVC(kernel='rbf', C=1.0, random_state=42)
        svm.fit(X_train, y_train)
        y_pred = svm.predict(X_test)
        
        print(f"\n--- SVM 欺诈检测结果 (下采样后欺诈比例: {y_res.mean():.4f}) ---")
        print(f"准确率: {accuracy_score(y_test, y_pred):.4f}")
        print(classification_report(y_test, y_pred))


        # 6.3 关联规则 (Apriori)
        # 目标：找出与欺诈相关的频繁项集
        df_rules = df_fraud.dropna(subset=['刷卡频率', '单笔是否透支', fraud_col]).copy()
        df_onehot = pd.get_dummies(df_rules[['刷卡频率', '单笔是否透支']].astype(str))
        
        # 添加欺诈项
        df_onehot['Fraud'] = df_rules[fraud_col].map(lambda x: 1 if '1' in str(x) or '有' in str(x) or '是' in str(x) else 0).astype(bool)
        
        frequent_itemsets = apriori(df_onehot, min_support=0.01, use_colnames=True)
        rules = association_rules(frequent_itemsets, metric='confidence', min_threshold=0.6)
        
        print("\n--- Apriori 关联规则 (置信度 > 0.6) ---")
        if not rules.empty:
            # 过滤后项包含 Fraud 的规则
            fraud_rules = rules[rules['consequents'].apply(lambda x: 'Fraud' in x)]
            print(fraud_rules.sort_values(by=['lift', 'confidence'], ascending=False).head(5))
        else:
            print("未找到满足条件的关联规则。")
    else:
        print("⚠️ 无法运行欺诈检测：消费表中缺少 '是否存在欺诈' 列。")


# ==================== 7. 步骤七：客户消费特征聚类 (KMeans) ====================
print("\n"+"="*40)
print("7. 步骤七：客户消费特征聚类 (KMeans)")
print("="*40)

if 'consume' in data_tables:
    df_clust = data_tables['consume'].copy()
    
    # 选择聚类字段
    cluster_cols = [c for c in ['日均消费金额','日均次数','单笔消费最小金额','单笔消费最高金额'] if c in df_clust.columns]
    
    if len(cluster_cols) >= 2:
        df_model = df_clust[cluster_cols].dropna()
        
        # 数值缩放
        scaler = StandardScaler()
        Xc = scaler.fit_transform(df_model)
        
        K = 2 # 假设 K=2 (报告中最常见的聚类数)
        kmeans = KMeans(n_clusters=K, random_state=42, n_init=10)
        kmeans.fit(Xc)
        
        # 评估 (轮廓系数)
        silhouette_avg = silhouette_score(Xc, kmeans.labels_)
        print(f" K={K} 聚类完成。轮廓系数: {silhouette_avg:.4f}")
        
        # 簇中心 (反缩放)
        centers = scaler.inverse_transform(kmeans.cluster_centers_)
        centers_df = pd.DataFrame(centers, columns=cluster_cols)
        centers_df['Count'] = df_model.groupby(kmeans.labels_).size()
        
        print("\n聚类簇中心 (反缩放，包含样本数):")
        display(centers_df)
    else:
        print("⚠️ 无法运行聚类分析：消费表中缺少足够的聚类特征。")

print("\n" + "="*40)
print("✅ 完整分析脚本执行完毕。")
print("="*40)

1. 步骤一：数据导入
✅ 数据导入成功。

2. 步骤二：数据合并与清洗 (已修复 Index 问题)
✅ 数据合并与清洗完成。

3. 步骤三：特征工程
✅ 衍生特征：'地区' 完成。


ValueError: cannot reindex on an axis with duplicate labels